In [0]:
df_source = spark.read.table("databricks_cat.silver.orders")
df_source.limit(10).display()

In [0]:
df_customer = spark.read.table("databricks_cat.gold.customers")
df_customer.limit(1).display()



In [0]:
df_products = spark.read.table("databricks_cat.gold.products")
df_products.limit(1).display()

In [0]:
df_join = df_source.join(df_customer, df_source.customer_id == df_customer.customer_id, "left").select(df_source["*"],df_customer["DimCustomerKey"])
df_join.limit(10).display()


In [0]:
df_join = df_join.join(df_products, df_join.product_id == df_products.product_id, "left").select(df_join["*"],df_products["DimProductKey"])


In [0]:
df_final = df_join.drop("customer_id", "product_id")
display(df_final.limit(1))

UPSERT THE Orders Table

In [0]:
from delta import DeltaTable

if spark.catalog.tableExists("databricks_cat.gold.orders"):
    df_target = DeltaTable.forName(spark, "databricks_cat.gold.orders")
    df_target.alias("trg").merge(df_final.alias("src"), "trg.order_id = src.order_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_final.write.mode("overwrite").format("delta").save("abfss://gold@databricksgvse2e.dfs.core.windows.net/orders")
    spark.sql("CREATE TABLE databricks_cat.gold.orders USING DELTA LOCATION 'abfss://gold@databricksgvse2e.dfs.core.windows.net/orders'")
